In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV


In [ ]:
df = pd.read_csv("Data/Football_Tableau.csv", low_memory=False)

df = df.replace(["?", "NA", "N/A", ""], np.nan)

df = df.dropna()

df.reset_index(drop=True, inplace=True)


features = [
    "EloDiff","Form3Diff","Form5Diff",
    "ShotDiff","TargetDiff",
    "FoulDiff","CornerDiff","CardDiff",
    "OddHome","OddDraw","OddAway"
]

X = df[features]
y = df["ResultLabel"]


In [ ]:
#Setup MLFlow tracking ui and experiment
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Football Match Outcome Prediction")

In [ ]:
with mlflow.start_run():

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))
    sample_weights = y_train.map(class_weights)

    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=7,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(xgb, X_train, y_train, cv=cv, scoring="f1_macro")

    xgb.fit(X_train, y_train, sample_weight=sample_weights)

    calibrated = CalibratedClassifierCV(xgb, method="isotonic", cv=3)
    calibrated.fit(X_train, y_train)

    preds = calibrated.predict(X_test)

    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", 900)
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("learning_rate", 0.03)

    mlflow.log_metric("cv_f1_mean", cv_scores.mean())
    mlflow.log_metric("test_f1_macro", f1)
    mlflow.log_metric("test_accuracy", acc)

from mlflow.models import infer_signature

signature = infer_signature(X_train, calibrated.predict(X_train))

mlflow.sklearn.log_model(
    calibrated,
    artifact_path="football_model",
    signature=signature,
    input_example=X_train.iloc[:5]
)

    print("CV F1:", cv_scores.mean())
    print("Test F1:", f1)
    print("Accuracy:", acc)
